# Posterior parameter recovery


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, PercentFormatter
from matplotlib_inline.backend_inline import set_matplotlib_formats
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401 -- registers the science style
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix
from sklearn.utils import resample

plt.style.use("science")
set_matplotlib_formats("svg")

FIGURE_STYLE = {
    "font.size": 17,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "axes.titlepad": 10,
    "axes.labelpad": 8,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "legend.title_fontsize": 16,
    "figure.titlesize": 22,
    "lines.linewidth": 2.0,
}
plt.rcParams.update(FIGURE_STYLE)

PARAMETER_LABELS = {
    "outside_optimism": "Outside-link optimism",
    "exploration": "Choice temperature",
    "visibility_distance": "Visibility radius",
}
PARAMETER_COLORS = ("#4477AA", "#EE6677", "#228833")
SQUARE_PANEL_SIZE_INCHES = 2.5
SQUARE_PANEL_GAP_INCHES = 0.5
SQUARE_PANEL_MARGINS_INCHES = (0.75, 0.25, 0.75, 0.5)


def square_panel_grid(
    n_rows,
    n_cols,
    *,
    margins=SQUARE_PANEL_MARGINS_INCHES,
    **subplot_kwargs,
):
    left, right, bottom, top = margins
    figure_width = (
        left
        + n_cols * SQUARE_PANEL_SIZE_INCHES
        + (n_cols - 1) * SQUARE_PANEL_GAP_INCHES
        + right
    )
    figure_height = (
        bottom
        + n_rows * SQUARE_PANEL_SIZE_INCHES
        + (n_rows - 1) * SQUARE_PANEL_GAP_INCHES
        + top
    )
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figure_width, figure_height),
        squeeze=False,
        subplot_kw={**(subplot_kwargs.pop("subplot_kw", None) or {}), "box_aspect": 1},
        **subplot_kwargs,
    )
    fig.subplots_adjust(
        left=left / figure_width,
        right=1 - right / figure_width,
        bottom=bottom / figure_height,
        top=1 - top / figure_height,
        wspace=SQUARE_PANEL_GAP_INCHES / SQUARE_PANEL_SIZE_INCHES,
        hspace=SQUARE_PANEL_GAP_INCHES / SQUARE_PANEL_SIZE_INCHES,
    )
    return fig, axes


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "network_creation"
        ).is_dir():
            return candidate
    raise RuntimeError("Could not locate the network-creation project root.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.model_inversion import load_posterior_artifact
from network_creation.model_inversion.diagnostics import (
    load_diagnostic_data,
    marginal_sbc_ranks,
    plot_sbc_ranks,
)
from network_creation.model_inversion.posterior import (
    category_probabilities,
    validate_discrete_theta_samples,
)
from network_creation.model_inversion.recovery_plots import (
    plot_posterior_ridgelines,
)

In [ ]:
import torch                                                                                                                                                                                                                             
                                                                                                                                                                                                                                              
torch.set_num_threads(20)                                                                                                                                                                                                
torch.set_num_interop_threads(1)

## 1. Data loading


In [ ]:
ARTIFACT_DIR = "mnpe-20260904T203340Z"

N_DIAGNOSTIC_CASES = None
N_POSTERIOR_SAMPLES_PER_CASE = 2**8
POSTERIOR_SAMPLE_CHUNK_SIZE = 256
MAX_SAMPLING_BATCH_SIZE = max(1, 100_000 // POSTERIOR_SAMPLE_CHUNK_SIZE)
MAX_SAMPLING_TIME_SECONDS = 300
RIDGELINE_CASES = 50
SEED = 42
DEVICE = "cpu"

In [ ]:
def resolve_artifact_dir(raw_path):
    path = Path(raw_path).expanduser()
    candidates = (
        (path,)
        if path.is_absolute()
        else (
            project_root / "artifacts" / "model_inversion" / path,
            project_root / path,
            Path.cwd() / path,
        )
    )
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"Posterior artifact not found. Searched:\n{searched}")

In [ ]:
artifact_dir = resolve_artifact_dir(ARTIFACT_DIR)
artifact = load_posterior_artifact(artifact_dir)
metadata = artifact.metadata
dataset_metadata = metadata.dataset
diagnostic_data = load_diagnostic_data(artifact)
prior_samples = artifact.prior_samples
theta_names = artifact.theta_order

diagnostic_input_values = artifact.input_values_from_x(diagnostic_data.x)
if "n_agents" not in diagnostic_input_values:
    raise ValueError(
        "The recovery views require n_agents in the artifact inputs."
    )
diagnostic_n_agents = diagnostic_input_values["n_agents"]

figure_output_dir = project_root / "artifacts" / "figures" / artifact_dir.name
figure_output_dir.mkdir(parents=True, exist_ok=True)

display(
    {
        "artifact_dir": (
            str(artifact_dir.relative_to(project_root))
            if artifact_dir.is_relative_to(project_root)
            else str(artifact_dir)
        ),
        "sbi_method": metadata.sbi.method,
        "posterior_type": type(artifact.posterior).__name__,
        "theta_order": theta_names,
        "continuous_theta_names": artifact.continuous_theta_names,
        "discrete_theta_names": artifact.discrete_theta_names,
        "x_shape": dataset_metadata.x_event_shape,
        "continuous_input_names": artifact.continuous_input_names,
        "discrete_input_names": artifact.discrete_input_names,
        "diagnostic_rows": int(diagnostic_data.theta.shape[0]),
        "stored_prior_samples": int(prior_samples.shape[0]),
    }
)

## 2. SBI fit quality


### Learning curve


In [ ]:
training_metrics_path = artifact_dir / metadata.sbi.training_metrics_file
training_metrics = json.loads(training_metrics_path.read_text(encoding="utf-8"))
loss_curves = {
    name: np.asarray(training_metrics[name], dtype=np.float64)
    for name in ("train_loss", "validation_loss")
    if training_metrics.get(name)
}
loss_summary = [
    {
        "metric": name,
        "n_epochs": int(values.size),
        "first": float(values[0]),
        "last": float(values[-1]),
        "best": float(values.min()),
    }
    for name, values in loss_curves.items()
]

if loss_curves:
    curve_labels = {
        "train_loss": "Training loss",
        "validation_loss": "Validation loss",
    }
    fig, (ax, summary_ax) = plt.subplots(
        2, 1, figsize=(10, 7), layout="constrained",
        gridspec_kw={"height_ratios": [5.5, 1]},
    )
    for color, (name, values) in zip(
        PARAMETER_COLORS,
        loss_curves.items(),
        strict=False,
    ):
        ax.plot(
            np.arange(1, len(values) + 1),
            values,
            color=color,
            label=curve_labels[name],
        )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("SBI training curve")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    summary_ax.set_axis_off()
    summary_table = summary_ax.table(
        cellText=[
            [curve_labels[row["metric"]], str(row["n_epochs"])]
            + [f"{row[field]:.6g}" for field in ("first", "last", "best")]
            for row in loss_summary
        ],
        colLabels=["Metric", "Epochs", "First", "Last", "Best (min)"],
        colWidths=[0.28, 0.12, 0.2, 0.2, 0.2],
        loc="center", cellLoc="center",
    )
    summary_table.auto_set_font_size(False)
    summary_table.set_fontsize(12)
    summary_table.scale(1, 1.6)
    figure_path = figure_output_dir / "learning_curve.pdf"
    fig.savefig(figure_path, format="pdf", bbox_inches="tight")
    print(f"Saved: {figure_path.relative_to(project_root)}")
    plt.show()
else:
    display(Markdown("No training or validation loss curves were stored."))

display(pd.DataFrame(loss_summary))

### Held-out posterior samples

Cases are sampled without replacement, stratified by `n_agents`, using scikit-learn. The seed makes selection reproducible, but selections and ordering differ from the previous custom NumPy sampler.

Posterior sampling delegates case iteration, chunking, and seed sequencing to `artifact.sample_cases`. A case-level progress bar shows completed cases, elapsed time, and estimated time remaining; SBI's internal sampling bars stay disabled.


In [ ]:
sample_size = len(diagnostic_n_agents)
if N_DIAGNOSTIC_CASES is not None:
    sample_size = min(int(N_DIAGNOSTIC_CASES), sample_size)
if sample_size <= 0:
    raise ValueError(
        "sample_size and the available diagnostic rows must be positive."
    )

diagnostic_indices = resample(
    np.arange(len(diagnostic_n_agents), dtype=np.int64),
    replace=False,
    n_samples=sample_size,
    stratify=diagnostic_n_agents,
    random_state=SEED,
)

In [ ]:
diagnostic_theta = validate_discrete_theta_samples(
    diagnostic_data.theta[diagnostic_indices],
    theta_names=artifact.theta_order,
    discrete_theta_values=artifact.schema.discrete_theta_values,
)
diagnostic_x = diagnostic_data.x[diagnostic_indices]
selected_n_agents = diagnostic_n_agents[diagnostic_indices]
n_agent_values = np.unique(selected_n_agents)

diagnostic_posterior_samples = artifact.sample_cases(
    diagnostic_x,
    num_samples=N_POSTERIOR_SAMPLES_PER_CASE,
    seed=SEED,
    device=DEVICE,
    max_samples_per_call=POSTERIOR_SAMPLE_CHUNK_SIZE,
    max_sampling_batch_size=MAX_SAMPLING_BATCH_SIZE,
    max_sampling_time=MAX_SAMPLING_TIME_SECONDS,
    show_case_progress=True,
    show_progress_bars=False,
)

group_rows = [
    {
        "n_agents": int(value),
        "available_cases": int(
            np.count_nonzero(diagnostic_n_agents == value)
        ),
        "selected_cases": int(np.count_nonzero(selected_n_agents == value)),
    }
    for value in n_agent_values
]
display(
    {
        "posterior_sample_shape": tuple(diagnostic_posterior_samples.shape),
        "requested_diagnostic_cases": N_DIAGNOSTIC_CASES,
        "available_diagnostic_cases": int(diagnostic_data.theta.shape[0]),
    }
)
display(pd.DataFrame(group_rows))

### Simulation-based calibration


In [ ]:
sbc_parameter_labels = [
    PARAMETER_LABELS.get(name, name.replace("_", " "))
    for name in theta_names
]


def style_sbc_axis(ax, *, title, show_legend):
    ax.set_title(title)
    ax.set_xlabel("Posterior rank bin")
    ax.set_ylabel("Empirical CDF")
    ax.grid(alpha=0.2)
    for artist in ax.get_children():
        if artist.get_label() == "expected under uniformity":
            artist.set_label("99% pointwise uniform-rank region")
    legend = ax.get_legend()
    if legend is None:
        return
    for text in legend.get_texts():
        if text.get_text() == "expected under uniformity":
            text.set_text("99% pointwise uniform-rank region")
    if not show_legend:
        legend.remove()
        return
    legend.set_title("Parameter")
    legend.set_frame_on(False)


ranks = marginal_sbc_ranks(
    artifact,
    diagnostic_posterior_samples,
    diagnostic_theta,
    seed=SEED,
)
sbc_data = {
    "ranks": ranks,
    "num_posterior_samples": N_POSTERIOR_SAMPLES_PER_CASE,
}
fig, axes = square_panel_grid(1, 1, margins=(0.75, 2.25, 0.75, 0.5), )
ax = axes[0, 0]
plot_sbc_ranks(
    artifact,
    sbc_data,
    parameter_labels=sbc_parameter_labels,
    colors=list(PARAMETER_COLORS),
    fig=fig,
    ax=ax,
    legend_kwargs={"loc": "center left", "bbox_to_anchor": (1.0, 0.5), "borderaxespad": 0, "frameon": False},
)
style_sbc_axis(
    ax,
    title=f"",
    show_legend=True,
)
plt.show()

In [ ]:
figure_path = figure_output_dir / "sbc_curve.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")

In [ ]:
n_sbc_rows = len(n_agent_values)
fig, axes = square_panel_grid(
    1,
    n_sbc_rows,
    sharex=True,
    sharey=True
)
for column_index, n_agents_value in enumerate(n_agent_values):
    group_mask = selected_n_agents == n_agents_value
    group_sbc_data = {
        "ranks": ranks[group_mask],
        "num_posterior_samples": N_POSTERIOR_SAMPLES_PER_CASE,
    }
    ax = axes[0, column_index]
    plot_sbc_ranks(
        artifact,
        group_sbc_data,
        parameter_labels=sbc_parameter_labels,
        colors=list(PARAMETER_COLORS),
        fig=fig,
        ax=ax,
        legend_kwargs={"frameon": False},
    )
    style_sbc_axis(
        ax,
        title=f"{n_agents_value:g} agents",
        show_legend=False,
    )
    if column_index != 0:
        ax.set_ylabel("")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.1),
    ncol=len(labels),
    frameon=False,
)
plt.show()

In [ ]:
figure_path = figure_output_dir / "sbc_curves_by_n_agents.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")

## 3. Parameter recovery

Ridgelines compare each held-out true parameter with its posterior marginal. Cases are randomly ordered by the stratified sampling step above; each panel then sorts them by the true value for readability.


In [ ]:
fig = plot_posterior_ridgelines(
    artifact,
    prior_samples=prior_samples,
    posterior_samples=diagnostic_posterior_samples[:RIDGELINE_CASES],
    true_theta=diagnostic_theta[:RIDGELINE_CASES],
)
figure_path = figure_output_dir / "posterior_ridgelines.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")
plt.show()

### Recovery by `n_agents`


In [ ]:
for n_agents_value in n_agent_values:
    group_mask = selected_n_agents == n_agents_value
    group_case_indices = np.flatnonzero(group_mask)[:RIDGELINE_CASES]
    display(
        Markdown(
            f"#### n_agents = {n_agents_value:g} "
            f"({len(group_case_indices)} displayed cases)"
        )
    )
    fig = plot_posterior_ridgelines(
        artifact,
        prior_samples=prior_samples,
        posterior_samples=diagnostic_posterior_samples[group_case_indices],
        true_theta=diagnostic_theta[group_case_indices],
        title=f"Posterior recovery: {n_agents_value:g} agents",
    )
    figure_path = figure_output_dir / f"posterior_ridgelines_n_agents_{int(n_agents_value)}.pdf"
    fig.savefig(figure_path, format="pdf", bbox_inches="tight")
    print(f"Saved: {figure_path.relative_to(project_root)}")
    plt.show()

## 4. Recovery matrix


In [ ]:
RECOVERY_MATRIX_LIMIT_PADDING = 0.01
RECOVERY_MATRIX_TICK_BINS = 6


def posterior_point_estimates(artifact, posterior_samples):
    samples = np.asarray(posterior_samples, dtype=np.float32)
    if samples.ndim != 3 or samples.shape[2] != len(artifact.theta_order):
        raise ValueError(
            "posterior_samples must have shape (case, sample, theta)."
        )

    estimates = np.full((samples.shape[0], samples.shape[2]), np.nan, dtype=np.float32)
    for theta_index, name in enumerate(artifact.theta_order):
        for case_index, values in enumerate(samples[:, :, theta_index]):
            if artifact.is_discrete_theta(name):
                categories = artifact.category_values(name)
                probabilities = category_probabilities(values, categories)
                estimates[case_index, theta_index] = categories[
                    int(np.argmax(probabilities))
                ]
                continue

            finite = values[np.isfinite(values)]
            if finite.size:
                estimates[case_index, theta_index] = np.mean(finite)
    return estimates


def pearson_label(x, y):
    finite = np.isfinite(x) & np.isfinite(y)
    if (
        finite.sum() < 2
        or np.isclose(np.std(x[finite]), 0.0)
        or np.isclose(np.std(y[finite]), 0.0)
    ):
        return "r = n/a"
    return f"r = {np.corrcoef(x[finite], y[finite])[0, 1]:.2f}"


def categorical_accuracy_label(true_values, mode_values):
    finite = np.isfinite(true_values) & np.isfinite(mode_values)
    if not np.any(finite):
        return "Accuracy = n/a"
    accuracy = np.mean(np.isclose(true_values[finite], mode_values[finite]))
    return f"Accuracy = {accuracy:.2f}"


def categorical_confusion_rates(true_values, mode_values, categories):
    pairs = np.column_stack((true_values, mode_values))
    # Map to category positions, preserving approximate matching and first matches.
    matches = np.isclose(categories, pairs[:, :, None])
    valid = np.isfinite(pairs).all(axis=1) & matches.any(axis=2).all(axis=1)
    if not np.any(valid):
        return np.zeros((len(categories), len(categories)), dtype=np.float32)
    positions = matches[valid].argmax(axis=2)
    # sklearn puts truth on rows; the recovery plot puts truth on columns.
    return confusion_matrix(
        positions[:, 0],
        positions[:, 1],
        labels=np.arange(len(categories)),
        normalize="true",
    ).T.astype(np.float32)


def plot_categorical_confusion(ax, true_values, mode_values, categories):
    rates = categorical_confusion_rates(true_values, mode_values, categories)
    image = ax.imshow(
        rates,
        vmin=0.0,
        vmax=1.0,
        cmap="Blues",
        aspect="equal",
        origin="lower",
    )
    labels = [f"{value:g}" for value in categories]
    ax.set_xticks(np.arange(len(categories)), labels=labels)
    ax.set_yticks(np.arange(len(categories)), labels=labels)
    ax.minorticks_off()
    ax.tick_params(axis="both", which="both", length=0)
    return image


def plot_recovery_matrix(
    artifact,
    *,
    theta_names,
    true_theta,
    posterior_estimates,
    n_agents,
):
    n_agent_values = np.unique(n_agents)
    column_labels = [
        PARAMETER_LABELS.get(
            name,
            name.rsplit(".", 1)[-1].replace("_", " ").capitalize(),
        )
        for name in theta_names
    ]

    continuous_limits = {}
    for theta_index, name in enumerate(theta_names):
        if artifact.is_discrete_theta(name):
            continue
        prior_low = float(artifact.metadata.dataset.prior_low[theta_index])
        prior_high = float(artifact.metadata.dataset.prior_high[theta_index])
        if np.isfinite(prior_low) and np.isfinite(prior_high) and prior_low < prior_high:
            padding = RECOVERY_MATRIX_LIMIT_PADDING * (prior_high - prior_low)
            continuous_limits[theta_index] = (
                prior_low - padding,
                prior_high + padding,
            )
            continue
        true_values = true_theta[:, theta_index]
        estimate_values = posterior_estimates[:, theta_index]
        finite = np.isfinite(true_values) & np.isfinite(estimate_values)
        if not np.any(finite):
            continue
        lower = float(min(true_values[finite].min(), estimate_values[finite].min()))
        upper = float(max(true_values[finite].max(), estimate_values[finite].max()))
        if lower == upper:
            lower, upper = lower - 0.5, upper + 0.5
        padding = RECOVERY_MATRIX_LIMIT_PADDING * (upper - lower)
        continuous_limits[theta_index] = (lower - padding, upper + padding)

    with plt.rc_context(FIGURE_STYLE):
        n_rows = len(n_agent_values)
        n_cols = len(theta_names)
        matrix_left_margin = 1.75
        fig, axes = square_panel_grid(
            n_rows,
            n_cols,
            margins=(matrix_left_margin, 1.5, 0.75, 1.25),
            sharex="col",
            sharey="col",
        )
        figure_width, figure_height = fig.get_size_inches()
        categorical_image = None

        for row_index, n_agents_value in enumerate(n_agent_values):
            group_mask = n_agents == n_agents_value
            for theta_index, name in enumerate(theta_names):
                ax = axes[row_index, theta_index]
                ax.tick_params(axis="x", which="both", labelbottom=True)
                ax.tick_params(axis="y", which="both", labelleft=True)
                true_values = true_theta[group_mask, theta_index]
                estimate_values = posterior_estimates[group_mask, theta_index]

                if artifact.is_discrete_theta(name):
                    categories = artifact.category_values(name).astype(np.float32)
                    categorical_image = plot_categorical_confusion(
                        ax,
                        true_values,
                        estimate_values,
                        categories,
                    )
                    label = categorical_accuracy_label(true_values, estimate_values)
                else:
                    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
                    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
                    finite = np.isfinite(true_values) & np.isfinite(estimate_values)
                    ax.scatter(
                        true_values[finite],
                        estimate_values[finite],
                        s=32,
                        alpha=0.35,
                        facecolor="gray",
                        edgecolor="none",
                    )
                    limits = continuous_limits.get(theta_index)
                    if limits is not None:
                        ax.set_xlim(limits)
                        ax.set_ylim(limits)
                        major_ticks = MaxNLocator(
                            nbins=RECOVERY_MATRIX_TICK_BINS
                        ).tick_values(*limits)
                        major_ticks = major_ticks[
                            (major_ticks >= limits[0]) & (major_ticks <= limits[1])
                        ]
                        ax.set_xticks(major_ticks)
                        ax.set_yticks(major_ticks)
                    label = pearson_label(true_values, estimate_values)
                    ax.grid(alpha=0.25)
                ax.text(
                    0.04,
                    0.96,
                    label,
                    transform=ax.transAxes,
                    va="top",
                    ha="left",
                    fontsize=12,
                    bbox={
                        "facecolor": "white",
                        "alpha": 0.75,
                        "edgecolor": "none",
                    },
                )

        strip_size_inches = 0.35
        strip_left = (
            matrix_left_margin
            - strip_size_inches
            - SQUARE_PANEL_GAP_INCHES
        ) / figure_width
        strip_width = strip_size_inches / figure_width
        for row_index, n_agents_value in enumerate(n_agent_values):
            position = axes[row_index, 0].get_position()
            fig.add_artist(
                Rectangle(
                    (strip_left, position.y0),
                    strip_width,
                    position.height,
                    transform=fig.transFigure,
                    facecolor="0.96",
                    edgecolor="0.75",
                    linewidth=0.8,
                )
            )
            fig.text(
                strip_left + strip_width / 2,
                position.y0 + position.height / 2,
                f"(n={n_agents_value:g})",
                ha="center",
                va="center",
                rotation=90,
                fontsize=15,
                fontweight="bold",
            )

        column_strip_bottom = axes[0, 0].get_position().y1 + (
            SQUARE_PANEL_GAP_INCHES / (4 * figure_height)
        )
        column_strip_height = strip_size_inches / figure_height
        for column_index, label in enumerate(column_labels):
            position = axes[0, column_index].get_position()
            fig.add_artist(
                Rectangle(
                    (position.x0, column_strip_bottom),
                    position.width,
                    column_strip_height,
                    transform=fig.transFigure,
                    facecolor="0.96",
                    edgecolor="0.75",
                    linewidth=0.8,
                )
            )
            fig.text(
                position.x0 + position.width / 2,
                column_strip_bottom + column_strip_height / 2,
                label,
                ha="center",
                va="center",
                fontsize=15,
                fontweight="bold",
            )

        if categorical_image is not None:
            colorbar_ax = fig.add_axes([0.93, 0.31, 0.018, 0.46])
            colorbar_ticks = np.linspace(0.0, 1.0, 5)
            colorbar = fig.colorbar(
                categorical_image,
                cax=colorbar_ax,
                ticks=colorbar_ticks,
                format=PercentFormatter(xmax=1, decimals=0),
            )
            colorbar.ax.set_ylabel(
                "Prediction rate within generating category",
                fontsize=15,
                rotation=90,
                ha="center",
                va="bottom",
                multialignment="center",
                labelpad=6,
            )
            colorbar.ax.yaxis.set_label_position("left")
            colorbar.ax.tick_params(labelsize=11, length=0)
            colorbar.ax.minorticks_off()
            colorbar.outline.set_linewidth(0.8)

        fig.supxlabel("Generating value or category", y=0.02)
        fig.supylabel("Posterior estimate or category", x=0.05)
        return fig, axes

In [ ]:
posterior_estimates = posterior_point_estimates(
    artifact,
    diagnostic_posterior_samples,
)

fig, axes = plot_recovery_matrix(
    artifact,
    theta_names=theta_names,
    true_theta=diagnostic_theta,
    posterior_estimates=posterior_estimates,
    n_agents=selected_n_agents,
)
plt.show()

In [ ]:
figure_path = figure_output_dir / "posterior_parameter_recovery_matrix.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")